# Türkçe LLM Fine-Tune: Unsloth + QLoRA + H100

**Model:** `AlicanKiraz0/Kizagan-E4B-Turkish-Reasoning-Model`  
**Hedef:** Function calling + genel Türkçe sohbet yeteneği kazandırma  
**Donanım:** NVIDIA H100 (Hopper mimarisi, bfloat16 + Flash Attention 2)

> Çalıştırmadan önce: Runtime → Change runtime type → **T4 GPU** yerine **A100/H100** seç.

In [ ]:
# ============================================================
# HÜCRE 1 — GPU Doğrulama
# H100 üzerinde çalıştığımızı teyit ediyoruz.
# ============================================================
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA mevcut: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Compute capability >= 8.0 → Ampere/Hopper → bf16 + Flash Attention 2 destekli
major, minor = torch.cuda.get_device_capability(0)
print(f'Compute Capability: {major}.{minor}')
assert major >= 8, 'bf16 için en az Ampere (8.x) gerekli!'

In [ ]:
# ============================================================
# HÜCRE 2 — Kurulum
# Unsloth'u Hopper (H100) uyumlu şekilde kuruyoruz.
# pip'in önbelleğini devre dışı bırakarak temiz kurulum.
# ============================================================
import sys

# Unsloth: cu124 = CUDA 12.4, torch260 = PyTorch 2.6.0 (Colab'ın mevcut versiyonuna göre ayarla)
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir "unsloth_zoo"

# Flash Attention 2 — H100'de zorunlu, hız farkı ~2-3x
!pip install --no-cache-dir flash-attn --no-build-isolation

# TRL (SFTTrainer), datasets, bitsandbytes
!pip install --no-cache-dir "trl>=0.9.0" "datasets>=2.18.0" "bitsandbytes>=0.43.0" "transformers>=4.40.0"

# GGUF export için llama.cpp Python bağlamaları
!pip install --no-cache-dir llama-cpp-python

print('\n✓ Kurulum tamamlandı.')

In [ ]:
# ============================================================
# HÜCRE 3 — Hugging Face Girişi
# Hub'a push_to_hub yapabilmek için token gerekli.
# Token: https://huggingface.co/settings/tokens (write izni)
# ============================================================
from huggingface_hub import login
from getpass import getpass

hf_token = getpass('HuggingFace Write Token: ')
login(token=hf_token)
print('✓ HuggingFace girişi başarılı.')

In [ ]:
# ============================================================
# HÜCRE 4 — Sabitler ve Yapılandırma
# Tüm parametreleri tek yerde tutuyoruz, bakımı kolaylaşır.
# ============================================================

# --- Model ---
BASE_MODEL_ID   = 'AlicanKiraz0/Kizagan-E4B-Turkish-Reasoning-Model'
OUTPUT_DIR      = './kizagan-finetuned-lora'   # LoRA adaptörleri buraya
HUB_MODEL_ID    = 'Tuguberk/Kizagan-E4B-FunctionCalling-TR'  # Hub'a yüklenecek ad

# --- Unsloth / Model yükleme ---
MAX_SEQ_LENGTH  = 8192    # Reasoning modelleri için geniş context
LOAD_IN_4BIT    = True    # QLoRA: 4-bit NF4 quantization
USE_BF16        = True    # H100 → bfloat16 (fp16'dan daha kararlı)

# --- LoRA ---
LORA_R          = 32      # Rank — yüksek rank = daha fazla parametre öğrenir
LORA_ALPHA      = 32      # Scaling: alpha/r = 1 → stabil başlangıç
LORA_DROPOUT    = 0.05
LORA_TARGETS    = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',  # FFN katmanları
]

# --- Dataset ---
HERMES_DS_ID    = 'Tuguberk/turkish-hermes-function-calling'
# 144K Türkçe prompt/completion çifti — catastrophic forgetting önleyici.
# Format: {'prompt': str, 'completion': str}, think/tool bloğu YOK.
GENERAL_DS_ID   = 'ogulcanaydogan/Turkish-LLM-v10-Training'
HERMES_RATIO    = 0.85    # %85 function-calling
GENERAL_RATIO   = 0.15    # %15 genel Türkçe sohbet
MAX_SAMPLES     = None    # None = tüm veri; hızlı test için 5000 koy

# --- Eğitim ---
BATCH_SIZE      = 4       # H100 80GB için 4-8 arası güvenli
GRAD_ACCUM      = 4       # Efektif batch = 4 x 4 = 16
LEARNING_RATE   = 2e-4
MAX_STEPS       = 2000    # Epoch yerine adım bazlı (dataset büyüklüğüne göre ayarla)
WARMUP_STEPS    = 100
SAVE_STEPS      = 200
LOGGING_STEPS   = 20
SEED            = 42

print('✓ Yapılandırma yüklendi.')
print(f'  Efektif batch boyutu: {BATCH_SIZE * GRAD_ACCUM}')
print(f'  Toplam adım: {MAX_STEPS}')

In [ ]:
# ============================================================
# HÜCRE 5 — Model ve Tokenizer Yükleme
# Unsloth'un FastLanguageModel'i:
#   - Flash Attention 2'yi otomatik aktif eder (H100'de)
#   - 4-bit NF4 quantization uygular
#   - bf16 ile gradient checkpoint'i birlikte yönetir
# ============================================================
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.bfloat16,   # H100 native dtype
    load_in_4bit   = LOAD_IN_4BIT,
    token          = hf_token,
    # Flash Attention 2 — Unsloth otomatik algılar, ama açıkça belirtelim
    attn_implementation = 'flash_attention_2',
)

print(f'✓ Model yüklendi: {BASE_MODEL_ID}')
print(f'  Parametre sayısı: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')
print(f'  dtype: {next(model.parameters()).dtype}')
print(f'  Tokenizer vocab size: {len(tokenizer)}')

In [ ]:
# ============================================================
# HÜCRE 6 — LoRA Adaptörlerini Ekle
# get_peft_model: modelin üstüne eğitilebilir LoRA katmanları ekler.
# Freeze edilmiş base model + küçük LoRA matrisleri = verimli fine-tune.
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r                    = LORA_R,
    lora_alpha           = LORA_ALPHA,
    lora_dropout         = LORA_DROPOUT,
    target_modules       = LORA_TARGETS,
    bias                 = 'none',
    use_gradient_checkpointing = 'unsloth',  # Unsloth'un özel optimize edilmiş versiyonu
    random_state         = SEED,
    use_rslora           = False,  # RSLoRA dene istersen — büyük r değerlerinde daha kararlı
    loftq_config         = None,
)

# Eğitilebilir parametre oranını göster
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✓ LoRA adaptörleri eklendi.')
print(f'  Eğitilebilir: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# ============================================================
# HÜCRE 7 — Chat Template İnceleme
# Modelin orijinal chat template'ini okuyup kaydediyoruz.
# Reasoning modelinin <think>...</think> yapısını korumak kritik!
# ============================================================

# Tokenizer'ın mevcut chat template'ini göster
print('=== Mevcut Chat Template ===')
if tokenizer.chat_template:
    print(tokenizer.chat_template[:1500], '...')
else:
    print('(Tanımlı template yok — ChatML kullanacağız)')

print()
print('=== Özel Token Listesi ===')
special_tokens = {
    'bos_token': tokenizer.bos_token,
    'eos_token': tokenizer.eos_token,
    'pad_token': tokenizer.pad_token,
    'unk_token': tokenizer.unk_token,
}
for k, v in special_tokens.items():
    print(f'  {k}: {repr(v)}')

In [ ]:
# ============================================================
# HÜCRE 8 — Chat Template Ayarla
# Kizagan modeli büyük ihtimalle ChatML ya da Qwen formatı kullanır.
# Reasoning (think) bloklarını koruyarak template'i standardize ediyoruz.
#
# Format:
#   <|im_start|>system\n{system}<|im_end|>
#   <|im_start|>user\n{user}<|im_end|>
#   <|im_start|>assistant\n<think>\n{reasoning}\n</think>\n{response}<|im_end|>
# ============================================================
from unsloth.chat_templates import get_chat_template

# Modelin kendi template'i varsa onu kullan, yoksa chatml'e düş
try:
    tokenizer = get_chat_template(
        tokenizer,
        chat_template = 'chatml',      # veya 'auto' dersen tokenizer'ı okur
        mapping = {                    # mevcut role adları farklıysa eşle
            'role'    : 'role',
            'content' : 'content',
            'user'    : 'user',
            'assistant': 'assistant',
            'system'  : 'system',
        },
    )
    print('✓ Chat template ayarlandı (ChatML).')
except Exception as e:
    print(f'get_chat_template hatası: {e}')
    print('Devam ediyoruz — tokenizer.apply_chat_template kullanacağız.')

# Pad token yoksa eos'u kullan (sol padding değil, sağ padding)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print('  pad_token → eos_token olarak ayarlandı.')

# Örnek format çıktısı
sample_messages = [
    {'role': 'system',    'content': 'Sen yardımcı bir Türkçe asistansın.'},
    {'role': 'user',      'content': 'Merhaba! Nasılsın?'},
    {'role': 'assistant', 'content': '<think>\nKullanıcı selamlama yapıyor.\n</think>\nMerhaba! İyiyim, teşekkürler.'},
]
formatted = tokenizer.apply_chat_template(sample_messages, tokenize=False)
print('\n=== Örnek Formatlı Mesaj ===')
print(formatted)

In [ ]:
# ============================================================
# HÜCRE 9 — Veri Setlerini Yükle
#
# Dataset 1 — Hermes (asıl görev):
#   Türkçe function-calling + reasoning örnekleri.
#   Format: conversations listesi ya da instruction/output çiftleri.
#
# Dataset 2 — Turkish-LLM-v10 (catastrophic forgetting kalkanı):
#   144K sade Türkçe soru-cevap. Format: {prompt, completion}.
#   Think/tool bloğu YOK — bunu normalize ederken dikkatli işleyeceğiz.
# ============================================================
from datasets import load_dataset, concatenate_datasets

print(f'Hermes dataset yükleniyor: {HERMES_DS_ID}...')
hermes_ds = load_dataset(HERMES_DS_ID, split='train', token=hf_token)
print(f'  ✓ {len(hermes_ds):,} örnek')
print(f'  Kolonlar: {hermes_ds.column_names}')
print()

print(f'Turkish-LLM-v10 dataset yükleniyor: {GENERAL_DS_ID}...')
general_ds = load_dataset(GENERAL_DS_ID, split='train')
print(f'  ✓ {len(general_ds):,} örnek')
print(f'  Kolonlar: {general_ds.column_names}')

# Yapı doğrulaması — beklenen: ['prompt', 'completion']
assert 'prompt' in general_ds.column_names, "Beklenen 'prompt' kolonu bulunamadı!"
assert 'completion' in general_ds.column_names, "Beklenen 'completion' kolonu bulunamadı!"
print('  ✓ Kolon yapısı doğrulandı: prompt / completion')

print('\n=== Hermes İlk Örnek ===')
print(hermes_ds[0])
print('\n=== Turkish-LLM-v10 İlk Örnek ===')
print(general_ds[0])
print('\n=== Turkish-LLM-v10 5. Örnek (uzun completion kontrolü) ===')
print(general_ds[4])

In [ ]:
# ============================================================
# HÜCRE 10 — Dataset Format Normalizer'ları
#
# SORUN: Turkish-LLM-v10 completionlarında <think> bloğu yok.
# Eğer bunları olduğu gibi assistant yanıtı olarak koyarsak,
# model "assistant bazen think olmadan yanıtlıyor" öğrenir ve
# base modeldeki reasoning formatı bozulabilir.
#
# ÇÖZÜM: Her sade completion'ın önüne boş bir <think> bloğu ekliyoruz:
#   <think>
#   </think>
#   {completion}
#
# Bu şekilde:
#   • Reasoning format bozulmaz (model think token'larını görmeye devam eder)
#   • Model "bu kadar basit bir soru için derin düşünmeye gerek yok" öğrenir
#   • Hermes örneklerindeki dolu think blokları baskın kalmaya devam eder
# ============================================================
import json

# Boş think bloğu sabiti — reasoning formatını korur
EMPTY_THINK = '<think>\n</think>\n'

def normalize_hermes(example):
    """
    Hermes formatı: conversations listesi (human/gpt veya user/assistant)
    ya da instruction/output çifti. Think blokları olduğu gibi korunur.
    """
    messages = []

    if 'conversations' in example:
        raw = example['conversations']
        if isinstance(raw, str):
            raw = json.loads(raw)
        for turn in raw:
            role = turn.get('from', turn.get('role', 'user'))
            role = {'human': 'user', 'gpt': 'assistant', 'system': 'system'}.get(role, role)
            messages.append({
                'role'   : role,
                'content': str(turn.get('value', turn.get('content', ''))),
            })

    elif 'messages' in example:
        messages = example['messages']

    elif 'instruction' in example:
        sys_prompt = example.get('system_prompt',
            'Sen yetenekli bir Türkçe yapay zeka asistanısın. '
            'Gerektiğinde fonksiyon çağrıları yapabilirsin.')
        messages = [
            {'role': 'system',    'content': sys_prompt},
            {'role': 'user',      'content': example['instruction']},
            {'role': 'assistant', 'content': example.get('output', example.get('response', ''))},
        ]
    else:
        return {'text': None}

    if not messages:
        return {'text': None}

    if messages[0]['role'] != 'system':
        messages.insert(0, {
            'role'   : 'system',
            'content': ('Sen yetenekli bir Türkçe yapay zeka asistanısın. '
                        'Gerektiğinde fonksiyon çağrıları yapabilirsin.'),
        })

    text = tokenizer.apply_chat_template(
        messages,
        tokenize              = False,
        add_generation_prompt = False,
    )
    return {'text': text}


def normalize_turkish_llm_v10(example):
    """
    ogulcanaydogan/Turkish-LLM-v10-Training için özel normalizer.

    Format: {'prompt': str, 'completion': str}
    — think bloğu yok, tool call yok, düz Türkçe soru-cevap.

    Strateji: completion'ın önüne boş <think> bloğu ekliyoruz.
    Bu, base modelin reasoning formatını korur ve modele
    "basit sorular için think gerekmez" sinyali verir.
    """
    prompt     = (example.get('prompt') or '').strip()
    completion = (example.get('completion') or '').strip()

    if not prompt or not completion:
        return {'text': None}

    # completion'da zaten think varsa olduğu gibi bırak
    if completion.startswith('<think>'):
        assistant_content = completion
    else:
        assistant_content = EMPTY_THINK + completion

    messages = [
        {
            'role'   : 'system',
            'content': 'Sen yardımcı bir Türkçe yapay zeka asistanısın.',
        },
        {
            'role'   : 'user',
            'content': prompt,
        },
        {
            'role'   : 'assistant',
            'content': assistant_content,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize              = False,
        add_generation_prompt = False,
    )
    return {'text': text}


# Normalizer çıktılarını test et
print('=== normalize_hermes örnek çıktısı ===')
if hermes_ds[0]:
    sample_hermes = normalize_hermes(hermes_ds[0])
    print(sample_hermes['text'][:600] if sample_hermes['text'] else '(boş)')

print('\n=== normalize_turkish_llm_v10 örnek çıktısı ===')
sample_general = normalize_turkish_llm_v10(general_ds[0])
print(sample_general['text'][:600] if sample_general['text'] else '(boş)')

print('\n✓ Normalizer\'lar tanımlandı ve test edildi.')
print(f'  Boş think bloğu formatı: {repr(EMPTY_THINK[:30])}...')

In [ ]:
# ============================================================
# HÜCRE 11 — Dataset Dönüştürme ve Karıştırma
# %85 Hermes + %15 Turkish-LLM-v10 → tek karışık dataset.
# Boş/hatalı örnekler filtrelenir.
# ============================================================
import random

print('Hermes dataset dönüştürülüyor...')
hermes_formatted = hermes_ds.map(
    normalize_hermes,
    num_proc = 4,
    desc     = 'Hermes format',
).filter(lambda x: x['text'] is not None and len(x['text']) > 50)
hermes_formatted = hermes_formatted.select_columns(['text'])
print(f'  ✓ {len(hermes_formatted):,} geçerli örnek')

print('Turkish-LLM-v10 dataset dönüştürülüyor...')
general_formatted = general_ds.map(
    normalize_turkish_llm_v10,   # prompt/completion → boş think + ChatML
    num_proc = 4,
    desc     = 'Turkish-LLM-v10 format',
).filter(lambda x: x['text'] is not None and len(x['text']) > 50)
general_formatted = general_formatted.select_columns(['text'])
print(f'  ✓ {len(general_formatted):,} geçerli örnek (144K\'dan)')

# %85/%15 oranını hesapla
hermes_size  = len(hermes_formatted)
general_size = int(hermes_size * (GENERAL_RATIO / HERMES_RATIO))
general_size = min(general_size, len(general_formatted))

general_sampled = general_formatted.shuffle(seed=SEED).select(range(general_size))

# Birleştir ve karıştır
mixed_dataset = general_sampled
from datasets import concatenate_datasets
mixed_dataset = concatenate_datasets([hermes_formatted, general_sampled])
mixed_dataset = mixed_dataset.shuffle(seed=SEED)

if MAX_SAMPLES and len(mixed_dataset) > MAX_SAMPLES:
    mixed_dataset = mixed_dataset.select(range(MAX_SAMPLES))

total = hermes_size + general_size
print(f'\n✓ Karışık dataset hazır:')
print(f'  Hermes (function-calling)  : {hermes_size:,} (%{100*hermes_size/total:.0f})')
print(f'  Turkish-LLM-v10 (genel TR) : {general_size:,} (%{100*general_size/total:.0f})')
print(f'  Toplam                     : {len(mixed_dataset):,}')
print()

# Hermes ve genel örnekleri karşılaştıralım
print('=== Karışık Dataset — Rastgele 2 Örnek ===')
for i in random.sample(range(len(mixed_dataset)), 2):
    print(f'\n[{i}]', mixed_dataset[i]['text'][:400], '...')

In [ ]:
# ============================================================
# HÜCRE 12 — Token Uzunluğu Analizi
# max_seq_length'i aşan örnekleri filtreleyerek OOM riskini azaltıyoruz.
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

print('Token uzunlukları hesaplanıyor (örnekleme ile)...')
sample_size = min(2000, len(mixed_dataset))
sample_indices = random.sample(range(len(mixed_dataset)), sample_size)

lengths = []
for idx in sample_indices:
    tokens = tokenizer(mixed_dataset[idx]['text'], return_length=True)['length'][0]
    lengths.append(tokens)

lengths = np.array(lengths)
print(f'  Ortalama: {lengths.mean():.0f} token')
print(f'  Medyan  : {np.median(lengths):.0f} token')
print(f'  P95     : {np.percentile(lengths, 95):.0f} token')
print(f'  Maks    : {lengths.max():.0f} token')
print(f'  > {MAX_SEQ_LENGTH} token: {(lengths > MAX_SEQ_LENGTH).mean()*100:.1f}%  (filtrelenecek)')

plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=50, color='steelblue', edgecolor='white')
plt.axvline(MAX_SEQ_LENGTH, color='red', linestyle='--', label=f'max_seq_length={MAX_SEQ_LENGTH}')
plt.xlabel('Token Sayısı')
plt.ylabel('Örnek Sayısı')
plt.title('Dataset Token Uzunluğu Dağılımı')
plt.legend()
plt.tight_layout()
plt.savefig('token_distribution.png', dpi=100)
plt.show()
print('✓ Grafik kaydedildi: token_distribution.png')

In [ ]:
# ============================================================
# HÜCRE 13 — SFTTrainer Kurulumu
# TRL'nin SFTTrainer'ı packing ve on-the-fly tokenization yapar.
# Packing: kısa örnekleri max_seq_length'e sığacak şekilde birleştirir,
# böylece GPU boşa çalışmaz → ~%30 daha hızlı eğitim.
# ============================================================
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# bf16 desteğini doğrula
assert is_bfloat16_supported(), 'Bu GPU bf16 desteklemiyor!'

training_args = SFTConfig(
    # --- Çıktı ---
    output_dir                  = OUTPUT_DIR,
    run_name                    = 'kizagan-finetuned',

    # --- Adım ve batch ---
    max_steps                   = MAX_STEPS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,

    # --- Optimizasyon ---
    learning_rate               = LEARNING_RATE,
    optim                       = 'adamw_8bit',   # bitsandbytes 8-bit optimizer
    lr_scheduler_type           = 'cosine',        # LR cosine decay
    warmup_steps                = WARMUP_STEPS,
    weight_decay                = 0.01,

    # --- Precision ---
    bf16                        = True,
    fp16                        = False,
    tf32                        = True,            # H100'de TF32 matmul hızlandırması

    # --- Bellek ---
    gradient_checkpointing      = True,
    dataloader_pin_memory       = True,
    dataloader_num_workers      = 4,

    # --- Kaydetme ---
    save_strategy               = 'steps',
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 3,              # En iyi 3 checkpoint'i sakla
    load_best_model_at_end      = False,          # Sadece son checkpoint yeterli

    # --- Loglama ---
    logging_steps               = LOGGING_STEPS,
    report_to                   = ['tensorboard'],

    # --- SFT özel ---
    max_seq_length              = MAX_SEQ_LENGTH,
    dataset_text_field          = 'text',
    packing                     = True,           # Kısa örnekleri birleştir
    seed                        = SEED,
)

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = mixed_dataset,
    args            = training_args,
)

print('✓ SFTTrainer hazır.')
print(f'  Efektif batch: {BATCH_SIZE * GRAD_ACCUM}')
print(f'  Toplam adım  : {MAX_STEPS}')
print(f'  Tahmini süre : ~{MAX_STEPS * BATCH_SIZE * GRAD_ACCUM * MAX_SEQ_LENGTH / 1e6:.0f}M token işlenecek')

In [ ]:
# ============================================================
# HÜCRE 14 — Eğitim Öncesi VRAM Kullanım Kontrolü
# Eğitim başlamadan önce bellek durumunu log'la.
# H100'ün 80GB VRAM'inin ne kadarını kullanacağımıza bakıyoruz.
# ============================================================
import gc

gc.collect()
torch.cuda.empty_cache()

vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
vram_used  = torch.cuda.memory_allocated(0) / 1e9
vram_free  = vram_total - vram_used

print(f'=== VRAM Durumu (Eğitim Öncesi) ===')
print(f'  Toplam : {vram_total:.1f} GB')
print(f'  Kullanılan: {vram_used:.1f} GB ({100*vram_used/vram_total:.0f}%)')
print(f'  Boş   : {vram_free:.1f} GB')

In [ ]:
# ============================================================
# HÜCRE 15 — EĞİTİM BAŞLAT
# Bu hücre uzun sürebilir (MAX_STEPS'e bağlı).
# Loss değerlerini tensorboard ile izleyebilirsin:
#   %load_ext tensorboard
#   %tensorboard --logdir ./kizagan-finetuned-lora
# ============================================================
import time

print('Eğitim başlıyor...')
start_time = time.time()

# resume_from_checkpoint=True ile önceki checkpoint'ten devam edebilirsin
train_result = trainer.train(resume_from_checkpoint=False)

elapsed = time.time() - start_time
print(f'\n✓ Eğitim tamamlandı!')
print(f'  Süre       : {elapsed/3600:.2f} saat')
print(f'  Final Loss : {train_result.training_loss:.4f}')

# Training metrikleri
print('\n=== Eğitim Metrikleri ===')
for k, v in train_result.metrics.items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# HÜCRE 16 — LoRA Adaptörlerini Kaydet
# Sadece adaptör ağırlıkları (küçük dosya, ~100-500MB)
# Base model olmadan paylaşılabilir veya birleştirilebilir.
# ============================================================
import os

# Yerel diske kaydet
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'✓ LoRA adaptörleri kaydedildi: {OUTPUT_DIR}')

# Dosya boyutlarını listele
total_size = 0
for f in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1e6
        total_size += size
        print(f'  {f}: {size:.1f} MB')
print(f'  Toplam: {total_size:.1f} MB')

# Google Drive'a da yedekle (Colab için)
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree(OUTPUT_DIR, '/content/drive/MyDrive/kizagan-lora')
# print('✓ Google Drive yedeği oluşturuldu.')

In [ ]:
# ============================================================
# HÜCRE 17 — Hugging Face Hub'a Yükle (LoRA Adaptörleri)
# Sadece adaptörleri yükle → küçük repo, hızlı paylaşım.
# ============================================================
model.push_to_hub(
    HUB_MODEL_ID,
    token          = hf_token,
    private        = False,     # True → özel repo
    safe_serialization = True,
)
tokenizer.push_to_hub(
    HUB_MODEL_ID,
    token = hf_token,
)
print(f'✓ Hub\'a yüklendi: https://huggingface.co/{HUB_MODEL_ID}')

In [ ]:
# ============================================================
# HÜCRE 18 — Hızlı Çıkarım Testi (LoRA ile)
# Eğitim sonrası modelin düzgün çalıştığını doğruluyoruz.
# ============================================================
from unsloth import FastLanguageModel

# Inference moduna geç (dropout devre dışı, hız artar)
FastLanguageModel.for_inference(model)

def generate_response(messages, max_new_tokens=512):
    """Verilen mesaj listesinden yanıt üret."""
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = 'pt',
    ).to('cuda')

    with torch.no_grad():
        output_ids = model.generate(
            input_ids         = input_ids,
            max_new_tokens    = max_new_tokens,
            temperature       = 0.7,
            top_p             = 0.9,
            do_sample         = True,
            repetition_penalty = 1.1,
            pad_token_id      = tokenizer.eos_token_id,
        )

    # Sadece yeni üretilen token'ları decode et
    new_tokens = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# Test 1: Genel sohbet
test_messages_1 = [
    {'role': 'system', 'content': 'Sen yardımcı bir Türkçe yapay zeka asistanısın.'},
    {'role': 'user',   'content': 'Python'da bir listedeki tekrar eden elemanları bul ve say.'},
]
print('=== Test 1: Genel Sohbet ===')
resp1 = generate_response(test_messages_1)
print(resp1)

# Test 2: Function calling
test_messages_2 = [
    {'role': 'system', 'content': 'Sen fonksiyon çağırma yapabilen bir Türkçe asistansın. Araçlar: get_weather(city: str)'},
    {'role': 'user',   'content': 'İstanbul\'da bugün hava nasıl?'},
]
print('\n=== Test 2: Function Calling ===')
resp2 = generate_response(test_messages_2)
print(resp2)

In [ ]:
# ============================================================
# HÜCRE 19 — Merged 16-bit SafeTensors Export
# LoRA adaptörlerini base modele birleştirir → tek seferde çalışan model.
# vLLM, Text Generation Inference gibi sunucular için idealdir.
# Disk gereksinimi: ~modelin tam boyutu (4B model ≈ 8-10GB @ fp16)
# ============================================================
MERGED_DIR = './kizagan-merged-16bit'

print('Merged 16-bit model kaydediliyor...')
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method = 'merged_16bit',  # LoRA + base = tek model, fp16
)
print(f'✓ Merged model kaydedildi: {MERGED_DIR}')

# Hub'a yükle (merged tam model)
MERGED_HUB_ID = HUB_MODEL_ID + '-merged'
model.push_to_hub_merged(
    MERGED_HUB_ID,
    tokenizer,
    save_method = 'merged_16bit',
    token       = hf_token,
    private     = False,
)
print(f'✓ Merged model Hub\'a yüklendi: https://huggingface.co/{MERGED_HUB_ID}')

In [ ]:
# ============================================================
# HÜCRE 20 — GGUF Export (Ollama / llama.cpp için)
# GGUF formatı: Ollama, LM Studio, llama-cpp-python ile çalışır.
# Q4_K_M: kalite/boyut dengesi için en yaygın tercih.
# Q8_0 : daha yüksek kalite, daha büyük dosya.
# f16  : quantization yok, en yüksek kalite.
# ============================================================
GGUF_DIR = './kizagan-gguf'

# Q4_K_M — Ollama/LM Studio için önerilen
print('GGUF Q4_K_M export başlıyor...')
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method = 'q4_k_m',
)
print(f'✓ GGUF Q4_K_M kaydedildi: {GGUF_DIR}')

# İsteğe bağlı: Q8_0 (daha iyi kalite)
# model.save_pretrained_gguf(GGUF_DIR + '-q8', tokenizer, quantization_method='q8_0')

# İsteğe bağlı: Hub'a GGUF yükle
# model.push_to_hub_gguf(
#     HUB_MODEL_ID + '-GGUF',
#     tokenizer,
#     quantization_method = ['q4_k_m', 'q8_0'],  # birden fazla format
#     token               = hf_token,
# )

# GGUF dosyalarını listele
import glob
gguf_files = glob.glob(f'{GGUF_DIR}/*.gguf') + glob.glob(f'{GGUF_DIR}/*.bin')
for f in gguf_files:
    size = os.path.getsize(f) / 1e9
    print(f'  {os.path.basename(f)}: {size:.2f} GB')

In [ ]:
# ============================================================
# HÜCRE 21 — vLLM Deployment Talimatları
# Bu hücre çalıştırılmaz — sunucuda nasıl deploy edileceğini gösterir.
# ============================================================

vllm_cmd = f"""
# vLLM ile merged modeli sunucu olarak başlat:

pip install vllm

# Seçenek A: HuggingFace'ten doğrudan
python -m vllm.entrypoints.openai.api_server \\
    --model {MERGED_HUB_ID} \\
    --dtype bfloat16 \\
    --max-model-len {MAX_SEQ_LENGTH} \\
    --tensor-parallel-size 1 \\
    --gpu-memory-utilization 0.9 \\
    --host 0.0.0.0 \\
    --port 8000

# Seçenek B: Yerel merged modelden
python -m vllm.entrypoints.openai.api_server \\
    --model {MERGED_DIR} \\
    --dtype bfloat16

# Ollama ile GGUF kullan:
# 1. Modelfile oluştur:
#    FROM ./kizagan-gguf/unsloth.Q4_K_M.gguf
#    TEMPLATE \"{{{{ .System }}}}\\n\\n{{{{ .Prompt }}}}\"
#    PARAMETER temperature 0.7
#
# 2. Modeli oluştur:
#    ollama create kizagan-turkish -f Modelfile
#
# 3. Çalıştır:
#    ollama run kizagan-turkish
"""

print(vllm_cmd)

In [ ]:
# ============================================================
# HÜCRE 22 — Özet ve Dosya Listesi
# Üretilen tüm çıktıları listele.
# ============================================================
print('=' * 55)
print('FINE-TUNE TAMAMLANDI — ÇIKTI ÖZETİ')
print('=' * 55)

outputs = [
    (OUTPUT_DIR,    'LoRA Adaptörleri (küçük, birleştirilebilir)'),
    (MERGED_DIR,    'Merged 16-bit SafeTensors (vLLM / TGI için)'),
    (GGUF_DIR,      'GGUF Q4_K_M (Ollama / llama.cpp için)'),
]

for path, desc in outputs:
    if os.path.exists(path):
        size_gb = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, fns in os.walk(path)
            for f in fns
        ) / 1e9
        print(f'\n✓ {desc}')
        print(f'  Yol : {path}')
        print(f'  Boyut: {size_gb:.2f} GB')
    else:
        print(f'\n✗ {desc} — {path} bulunamadı')

print(f'\nHuggingFace Hub:')
print(f'  LoRA    : https://huggingface.co/{HUB_MODEL_ID}')
print(f'  Merged  : https://huggingface.co/{MERGED_HUB_ID}')
print()
print('Bir sonraki adım: vLLM veya Ollama ile deploy et (Hücre 21).')